# Alpha Engine — Colab E2 / US87 Model Batch (owner-operated)

This is a **real multi-trial XGBoost validation worker**, not a browser research IDE, hosted scheduler, GPU guarantee or strategy auto-publisher.

**Default: does not mount Drive, copy private data, train, or upload results.** Run only after reviewing actual source-provider rights for private Google Colab processing, your chosen immutable v5 market data and the complete 40-character CI-green experiment revision.

Reuses existing Study Workspace, US87 daily prices and QQQ. A validation search is NOT an untouched final-test evaluation. US Alpha158, PIT and trade-readiness remain blocked/unproven. **Never run the older alpha_research_exchange_v1 notebook against the v5 ZIP.**

Scope & code: [Issue #1373](https://github.com/liuh886/alpha_engine/issues/1373), stacked [PR #1374](https://github.com/liuh886/alpha_engine/pull/1374).

In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys

# Configure paths in your *private* Drive. No folder IDs, tokens, or sensitive
# metadata should be committed to GitHub.
V5_DIR = Path('/content/drive/MyDrive/CHANGE_ME/private/input_packages/us87-cn130-20261103-local-v5')
BATCH_SPEC_FILE = Path('/content/drive/MyDrive/CHANGE_ME/private/us87_xgb_batch.json')
CLOUD_RIGHTS_RECORD = Path('/content/drive/MyDrive/CHANGE_ME/private/us87_cloud_rights.json')
CHECKPOINT_PARENT = Path('/content/drive/MyDrive/CHANGE_ME/private/incoming_unreviewed')

EXPECTED_MANIFEST_SHA = 'd9f4fb2ef7aa2735cfb95436e7eabb744ffb8d6d387c4294626893fc262dab0c'
EXPECTED_US_HISTORY_SHA = '93b2d42cebf3ebfac7b80298f6822fdc4305f7550b73ac1b2bb8d272d3fb3de8'
VERIFIER_AND_MODEL_RUNNER_GIT_SHA = 'CHANGE_ME_EXACT_PR1374_COMMIT_SHA'
EXPECTED_BATCH_SPEC_SHA256 = 'CHANGE_ME_64_CHARACTER_BATCH_FILE_SHA256'
EXPECTED_CLOUD_RIGHTS_SHA256 = 'CHANGE_ME_64_CHARACTER_RIGHTS_FILE_SHA256'
UV_VERSION = '0.9.10'
PYTHON_VERSION = '3.12.12'

OPERATOR_APPROVES_PRIVATE_CLOUD_DATA_AND_TRAINING = False
RUN_ONE_TRAINING_CHUNK = False
EXPORT_IMMUTABLE_CHECKPOINT = False
CHUNK_TRIAL_BUDGET = 2

# Optional exact folder from a previous immutable checkpoint; never scan
# unattended or automatically select a "latest" checkpoint.
RESUME_FROM_VERIFIED_CHECKPOINT = None
EXPECTED_CHECKPOINT_INDEX_SHA256 = None

assert OPERATOR_APPROVES_PRIVATE_CLOUD_DATA_AND_TRAINING, 'Owner opt-in required'
assert all('CHANGE_ME' not in str(p) for p in
           (V5_DIR, BATCH_SPEC_FILE, CLOUD_RIGHTS_RECORD, CHECKPOINT_PARENT))
assert re.fullmatch(r'[a-f0-9]{40}', VERIFIER_AND_MODEL_RUNNER_GIT_SHA)
assert 1 <= CHUNK_TRIAL_BUDGET <= 20
assert re.fullmatch(r'[a-f0-9]{64}', EXPECTED_BATCH_SPEC_SHA256)
assert re.fullmatch(r'[a-f0-9]{64}', EXPECTED_CLOUD_RIGHTS_SHA256)

## 1 · Rights review and private Drive mount

A provider-specific authorization record must be reviewed by the owner. Permission to mount Drive is **not** permission from Yahoo or any other upstream price-data source. If the intended cloud computation is not licensed, stop before mounting/copying and use local-only execution.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert hashlib.sha256(CLOUD_RIGHTS_RECORD.read_bytes()).hexdigest() == EXPECTED_CLOUD_RIGHTS_SHA256
rights = json.loads(CLOUD_RIGHTS_RECORD.read_text(encoding='utf-8'))
assert rights.get('schema_version') == 'e1a_price_cloud_rights_v1'
assert rights.get('market') == 'us'
assert rights.get('bundle_manifest_sha256') == EXPECTED_MANIFEST_SHA
assert rights.get('history_archive_sha256') == EXPECTED_US_HISTORY_SHA
assert rights.get('cloud_processing_purpose') == 'private_colab_daily_price_research'
assert rights.get('owner_approved_private_cloud_processing') is True
assert rights.get('provider_rights') and all(
    p.get('private_cloud_compute_permitted') is True and
    str(p.get('terms_url', '')).startswith('https://') and
    len(str(p.get('review_basis', '')).strip()) >= 12
    for p in rights['provider_rights']
)
print('Owner-attested rights record present (not independent vendor certification).')
assert {p['provider'] for p in rights['provider_rights']} >= {'yfinance'}
print('Documentary proof path:', rights.get('documentary_proof_path') or 'Not supplied; owner attestation only.')

## 2 · Copy exactly the authorized price input once, then pin code and data

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

scratch = Path('/content/p2_e2_scratch')
scratch.mkdir(parents=True, exist_ok=False)
for src, name, expected in [
    (V5_DIR/'bundle-manifest.json', 'bundle-manifest.json', EXPECTED_MANIFEST_SHA),
    (V5_DIR/'us_market_history.zip', 'us_market_history.zip', EXPECTED_US_HISTORY_SHA)
]:
    dest = scratch/name
    shutil.copyfile(src, dest)
    assert sha256_file(dest) == expected, f'external byte pin mismatch: {name}'
batch_file = scratch/'batch.json'
shutil.copyfile(BATCH_SPEC_FILE, batch_file)
assert sha256_file(batch_file) == EXPECTED_BATCH_SPEC_SHA256
batch = json.loads(batch_file.read_text(encoding='utf-8'))
assert batch['runner_git_sha'] == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
assert batch['v5_bundle_manifest_sha256'] == EXPECTED_MANIFEST_SHA
assert batch['history_archive_sha256'] == EXPECTED_US_HISTORY_SHA
assert batch['purpose'] == 'validation_search_only'

checkout = Path('/content/p2_e2_checkout')
subprocess.run(['git','clone','--quiet','--filter=blob:none','--no-checkout',
                'https://github.com/liuh886/alpha_engine.git',str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'sparse-checkout','init','--cone'],check=True)
subprocess.run(['git','-C',str(checkout),'sparse-checkout','set','src'],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',
                VERIFIER_AND_MODEL_RUNNER_GIT_SHA],check=True)
assert subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],
                               text=True).strip() == VERIFIER_AND_MODEL_RUNNER_GIT_SHA
subprocess.run([sys.executable,'-m','pip','install','--quiet',f'uv=={UV_VERSION}'],check=True)
subprocess.run(['uv','sync','--frozen','--python',PYTHON_VERSION],cwd=checkout,check=True)
# Model work runs inside the frozen uv project, not Colab's ambient kernel.
environment_probe = "import json,platform,sys,os,importlib.metadata as m;print(json.dumps({'python':platform.python_version(),'platform':platform.platform(),'cpu_count':os.cpu_count(),'packages':{p:m.version(p) for p in ['numpy','pandas','scipy','xgboost']}}))"
environment = json.loads(subprocess.check_output(
    ['uv','run','--frozen','python','-c',environment_probe],cwd=checkout,text=True))
assert environment['python'] == PYTHON_VERSION
print(json.dumps(environment,indent=2))


## 3 · Verify v5 bytes and build an isolated, non-Git research data root

The original dataset remains immutable and local-only. Cloud permission is a separate private operator record, never a mutation to v5.

In [ ]:
def uv_module(*args):
    result = subprocess.run(['uv','run','--frozen','python','-m',*map(str,args)],
                            cwd=checkout,capture_output=True,text=True)
    if result.returncode:
        print(result.stderr[-4000:])
        result.check_returncode()
    payload = json.loads(result.stdout)
    print(json.dumps({k:v for k,v in payload.items() if k not in ('trials','receipt')},indent=2))
    return payload

uv_module('src.research.e1a_price_preflight',
          '--manifest',scratch/'bundle-manifest.json',
          '--archive',scratch/'us_market_history.zip',
          '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
          '--expected-archive-sha256',EXPECTED_US_HISTORY_SHA,
          '--rights-record',CLOUD_RIGHTS_RECORD,'--require-cloud-rights')
stage_root = scratch/'isolated_us87'
uv_module('src.research.v5_isolated_staging','stage',
          '--manifest',scratch/'bundle-manifest.json',
          '--archive',scratch/'us_market_history.zip',
          '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
          '--expected-archive-sha256',EXPECTED_US_HISTORY_SHA,
          '--output-root',stage_root)
uv_module('src.research.colab_experiment_batch','plan',
          '--batch-spec',batch_file)
print('Verified complete US87 input; no model execution yet.')

## 4 · Optional: restore one explicit, immutable prior checkpoint

Only import from a trusted private `incoming_unreviewed` folder after checking exact file hashes against its receipt. The local run identity, data and model Git SHA are still checked by the batch runner on `--resume`.
The checkpoint index SHA256 is independently required; do not infer it from an untrusted current folder. A mismatched or injected snapshot is rejected before restoring any output.

In [ ]:
output_root = scratch/'runs'
resume_flag = False
last_checkpoint = None
if RESUME_FROM_VERIFIED_CHECKPOINT is not None:
    assert re.fullmatch(r'[a-f0-9]{64}', str(EXPECTED_CHECKPOINT_INDEX_SHA256))
    uv_module('src.research.colab_experiment_batch','restore',
              '--batch-spec',batch_file,'--data-root',stage_root,
              '--checkpoint-root',Path(RESUME_FROM_VERIFIED_CHECKPOINT),
              '--expected-index-sha256',EXPECTED_CHECKPOINT_INDEX_SHA256,
              '--output-root',output_root)
    resume_flag = True


def save_checkpoint():
    import datetime
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest = CHECKPOINT_PARENT/('e2-us87-checkpoint-'+stamp)
    return uv_module('src.research.colab_experiment_batch','checkpoint',
                     '--batch-spec',batch_file,'--data-root',stage_root,
                     '--output-root',output_root,'--destination',dest)


## 5 · Run one bounded model-training chunk

Each invocation fits at most `CHUNK_TRIAL_BUDGET` new trials. Do not run this cell unless you have a provider-rights-approved cloud compute session and have inspected the frozen study spec. This is a validation search; no automatic final holdout or trading.

In [ ]:
if RUN_ONE_TRAINING_CHUNK:
    args = ['src.research.colab_experiment_batch','run',
            '--batch-spec',batch_file,'--data-root',stage_root,
            '--output-root',output_root,'--max-new-trials',CHUNK_TRIAL_BUDGET]
    if resume_flag:
        args.append('--resume')
    result = uv_module(*args)
    resume_flag = True
    if EXPORT_IMMUTABLE_CHECKPOINT:
        last_checkpoint = save_checkpoint()
        print('Save this checkpoint path and independent index SHA256 before resetting the runtime.')
else:
    print('No model fit: RUN_ONE_TRAINING_CHUNK is False.')


## 6 · Inspect the private, unreviewed checkpoint

With the separate export opt-in enabled, each completed training chunk immediately saves a fresh, hash-verified checkpoint through the same CLI used locally. It accepts only the frozen batch plan and verified trial specs/study receipts, with a 20 MiB bound. Raw data, extra files, changed code/batch identities and hash mismatches are refused before restore writes.

Record the returned checkpoint folder and `index_sha256` separately. In a fresh runtime, set both restore variables in the first cell and run the notebook again. Completed trials are skipped; an atomic completed study interrupted before its trial wrapper is recovered without fitting it twice. Results remain `incoming_unreviewed`.


In [ ]:
if last_checkpoint is not None:
    print(json.dumps(last_checkpoint,indent=2))
else:
    print('No checkpoint exported. Enable EXPORT_IMMUTABLE_CHECKPOINT before training to preserve completed chunks.')


## Acceptance notes

Do not claim Colab success until real model training, completed/blocked trial receipts, identical local baseline and actual resource observations exist. After the batch finishes, use the existing local Agent study review to inspect every run and cost assumption. Explicitly reserve a **separate, untouched final test** after hyperparameter selection. This notebook cannot certify licensing, point-in-time availability, survivorship-free universe selection or trade readiness.

The runtime is pinned to uv 0.9.10 and Python 3.12.12; code uses a sparse checkout and execution uses the frozen uv environment. Package versions, measured trial seconds and actual training row/session counts are retained. Hardware/performance equivalence still requires an owner-run Colab receipt. [Official Colab FAQ](https://research.google.com/colaboratory/faq.html).